# 🏆 TSTR/TRTS Model-Based Validation Framework

**The Gold Standard for Validating Augmented Data Quality**

This notebook implements the scientifically rigorous TSTR/TRTS framework to validate augmented/synthetic data quality.

---

## 📚 What is TSTR/TRTS?

| Method | Concept | What it Proves |
|--------|---------|----------------|
| **TSTR** (Train on Synthetic, Test on Real) | Train a model only on augmented/synthetic data, then test on real, held-out data | **Usability**: If the model learns generalized patterns from augmentation that apply to the real world, your data is high quality |
| **TRTS** (Train on Real, Test on Synthetic) | Train a model on real data and test it on your augmented data | **Realism**: If the model fails on augmented data, your augmentation has drifted too far from the real distribution (manifold) |

---

## 🎯 Gap Analysis

- **Small TSTR-TRTS gap** → High quality augmentation
- **Large gap** → Distribution mismatch between real and synthetic data

---

**Let's get started!**


## 🔧 Setup & Installation


In [ ]:
# Install required packages (run this in Colab)
# !pip install -q transformers datasets torch scikit-learn pandas numpy matplotlib seaborn underthesea


In [ ]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally")


In [ ]:
import os
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Underthesea for Vietnamese word segmentation
from underthesea import word_tokenize

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ Underthesea word tokenizer loaded")


## ⚙️ Configuration


In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these paths and parameters
# ============================================================================

# Data paths
if IN_COLAB:
    DATA_DIR = '/content/drive/MyDrive/thesis/data/processed/'
    OUTPUT_DIR = '/content/drive/MyDrive/thesis/tstr_trts_results'
else:
    DATA_DIR = './data/processed'
    OUTPUT_DIR = './tstr_trts_results'

# Data files - Combined data (original + augmented already merged)
COMBINED_TRAIN_PATH = os.path.join(DATA_DIR, 'train_1071_gen_final.csv')  # Combined original + augmented data
TEST_PATH = os.path.join(DATA_DIR, 'test_processed.csv')                   # Held-out test data

# Model configuration
MODEL_NAME = 'uitnlp/CafeBERT'  # CafeBERT - Vietnamese BERT model
MAX_LENGTH = 256

# Tokenization configuration
USE_UNDERTHESEA = False  # Use default tokenizer for CafeBERT (no word segmentation needed)

# Training configuration
NUM_EPOCHS = 3
BATCH_SIZE = 128
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
RANDOM_SEED = 42

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Configuration:")
print(f"   Combined train data: {COMBINED_TRAIN_PATH}")
print(f"   Test data: {TEST_PATH}")
print(f"   Model: {MODEL_NAME}")
print(f"   Use Underthesea: {USE_UNDERTHESEA}")
print(f"   Output: {OUTPUT_DIR}")


In [ ]:
# ============================================================================
# TEST TOKENIZER & MODEL
# ============================================================================
# Test CafeBERT tokenizer (no word segmentation needed)

from transformers import AutoModel, AutoTokenizer
from underthesea import word_tokenize as underthesea_word_tokenize
import torch

# Load model and tokenizer
model = AutoModel.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test text
test_text = 'Cà phê được trồng nhiều ở khu vực Tây Nguyên của Việt Nam.'

# ============================================================================
# TOKENIZER OPTION: Switch between 'default' and 'underthesea'
# CafeBERT works best with default tokenizer (no word segmentation)
# ============================================================================
TOKENIZER_MODE = 'default'  # Options: 'default' (recommended for CafeBERT) or 'underthesea'

def tokenize_text(text: str, mode: str = 'underthesea') -> str:
    """
    Tokenize text based on selected mode.
    
    Args:
        text: Input Vietnamese text
        mode: 'default' - use PhoBERT tokenizer directly
              'underthesea' - apply Underthesea word segmentation first (recommended for PhoBERT)
    
    Returns:
        Processed text ready for BERT tokenization
    """
    if mode == 'underthesea':
        # Apply Underthesea word segmentation (e.g., "học sinh" -> "học_sinh")
        return underthesea_word_tokenize(text, format="text")
    else:
        # Use text as-is for PhoBERT tokenizer
        return text

# Test both tokenization modes
print("🧪 Testing PhoBERT-large Tokenizer Modes...")
print("=" * 60)

for mode in ['default', 'underthesea']:
    print(f"\n📌 Mode: {mode.upper()}")
    print("-" * 40)
    
    processed_text = tokenize_text(test_text, mode=mode)
    print(f"Input text:     {test_text}")
    print(f"Processed text: {processed_text}")
    
    encoding = tokenizer(processed_text, return_tensors='pt')
    print(f"Token IDs:      {encoding['input_ids']}")
    print(f"Num tokens:     {encoding['input_ids'].shape[1]}")

# Test model inference with selected mode
print(f"\n{'=' * 60}")
print(f"🔧 Using TOKENIZER_MODE = '{TOKENIZER_MODE}' for experiments")
print("=" * 60)

processed_text = tokenize_text(test_text, mode=TOKENIZER_MODE)
encoding = tokenizer(processed_text, return_tensors='pt')

with torch.no_grad():
    output = model(**encoding)

print(f"\n📊 Model Output:")
print(f"   Last hidden state shape: {output.last_hidden_state.shape}")
print(f"   Pooler output shape: {output.pooler_output.shape}")
print(f"✅ {MODEL_NAME} tokenizer and model test passed!")

## 📊 Data Classes & Utilities


In [ ]:
@dataclass
class ValidationResult:
    """Container for validation experiment results"""
    experiment_name: str
    train_source: str  # 'real', 'synthetic', or 'mixed'
    test_source: str   # 'real' or 'synthetic'
    accuracy: float
    precision_weighted: float
    recall_weighted: float
    f1_weighted: float
    f1_macro: float
    per_class_metrics: Dict[str, Dict[str, float]]
    confusion_matrix: np.ndarray
    training_time: float
    num_train_samples: int
    num_test_samples: int


class EmotionDataset(Dataset):
    """PyTorch Dataset for emotion classification with optional Underthesea tokenization"""
    
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128, use_underthesea: bool = False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_underthesea = use_underthesea
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Apply Underthesea word segmentation if enabled
        if self.use_underthesea:
            text = tokenize_text(text, mode='underthesea')
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


# Test tokenization modes
print("🔤 Tokenization example:")
sample_text = "Tôi rất vui khi được học tiếng Việt"
print(f"   Original:    {sample_text}")
print(f"   Default:     {tokenize_text(sample_text, mode='default')}")
print(f"   Underthesea: {tokenize_text(sample_text, mode='underthesea')}")


## 🔬 TSTR/TRTS Validator Class


In [ ]:
class TSTRTRTSValidator:
    """
    TSTR/TRTS Validation Framework for Augmented Data Quality Assessment
    
    This class provides comprehensive validation of augmented data using
    model-based evaluation techniques.
    
    NOTE: This version expects combined data (original + augmented already merged).
    """
    
    def __init__(
        self,
        model_name: str = 'vinai/phobert-large',
        output_dir: str = './tstr_trts_results',
        max_length: int = 128,
        random_seed: int = 42,
        use_underthesea: bool = True  # Required for PhoBERT
    ):
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        self.random_seed = random_seed
        self.use_underthesea = use_underthesea
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"🔧 TSTR/TRTS Validator initialized")
        print(f"   Model: {model_name}")
        print(f"   Device: {self.device}")
        print(f"   Use Underthesea: {use_underthesea}")
        
        os.makedirs(output_dir, exist_ok=True)
        
        self.tokenizer = None
        self.label2id = None
        self.id2label = None
        self.results: List[ValidationResult] = []
    
    def _setup_label_mappings(self, df: pd.DataFrame, emotion_col: str = 'Emotion'):
        """Setup label to ID mappings from data"""
        unique_labels = sorted(df[emotion_col].unique())
        self.label2id = {label: idx for idx, label in enumerate(unique_labels)}
        self.id2label = {idx: label for label, idx in self.label2id.items()}
        print(f"   Labels: {list(self.label2id.keys())}")
    
    def load_combined_data(
        self,
        combined_data_path: str,
        test_data_path: str
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Load combined (original + augmented) training data and test data.
        
        Args:
            combined_data_path: Path to combined training data (original + augmented)
            test_data_path: Path to held-out test data
        
        Returns:
            Tuple of (combined_train_df, test_df)
        """
        print("\n📊 Loading datasets...")
        
        # Label mapping: Vietnamese -> English (for consistency)
        vn_to_en = {
            'vui vẻ': 'Enjoyment', 'khó chịu': 'Disgust', 'khác': 'Other',
            'buồn': 'Sadness', 'giận dữ': 'Anger', 'sợ hãi': 'Fear',
            'ngạc nhiên': 'Surprise', 'buồn bã': 'Sadness'
        }
        
        # Load combined training data
        combined_df = pd.read_csv(combined_data_path)
        print(f"   Combined training data: {len(combined_df)} samples")
        
        # Detect column names and standardize
        if 'Sentence_clean' in combined_df.columns:
            text_col = 'Sentence_clean'
        elif 'Sentence' in combined_df.columns:
            text_col = 'Sentence'
        else:
            raise ValueError(f"Cannot detect text column. Columns: {combined_df.columns.tolist()}")
        
        if 'Emotion' in combined_df.columns:
            emotion_col = 'Emotion'
        elif 'emotion_vn' in combined_df.columns:
            emotion_col = 'emotion_vn'
        else:
            raise ValueError(f"Cannot detect emotion column. Columns: {combined_df.columns.tolist()}")
        
        combined_df = combined_df.rename(columns={text_col: 'text', emotion_col: 'label'})
        combined_df['label'] = combined_df['label'].apply(lambda x: vn_to_en.get(x, x))
        
        # Load test data
        test_df = pd.read_csv(test_data_path)
        print(f"   Test data: {len(test_df)} samples")
        
        if 'Sentence' in test_df.columns:
            test_df = test_df.rename(columns={'Sentence': 'text', 'Emotion': 'label'})
        elif 'Sentence_clean' in test_df.columns:
            test_df = test_df.rename(columns={'Sentence_clean': 'text'})
            if 'Emotion' in test_df.columns:
                test_df = test_df.rename(columns={'Emotion': 'label'})
            elif 'emotion_vn' in test_df.columns:
                test_df = test_df.rename(columns={'emotion_vn': 'label'})
        
        test_df['label'] = test_df['label'].apply(lambda x: vn_to_en.get(x, x))
        
        # Setup label mappings from combined data
        all_labels = pd.concat([combined_df['label'], test_df['label']])
        label_df = pd.DataFrame({'Emotion': all_labels})
        self._setup_label_mappings(label_df, 'Emotion')
        
        print(f"   Unique labels: {list(self.label2id.keys())}")
        
        return combined_df, test_df
    
    def _create_dataset(self, df: pd.DataFrame, text_col: str = 'text', label_col: str = 'label') -> EmotionDataset:
        """Create PyTorch dataset from DataFrame with optional Underthesea tokenization"""
        texts = df[text_col].tolist()
        labels = [self.label2id[label] for label in df[label_col]]
        return EmotionDataset(texts, labels, self.tokenizer, self.max_length, use_underthesea=self.use_underthesea)
    
    def _compute_metrics(self, eval_pred) -> Dict[str, float]:
        """Compute metrics for Trainer"""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )
        f1_macro = precision_recall_fscore_support(
            labels, predictions, average='macro', zero_division=0
        )[2]
        accuracy = accuracy_score(labels, predictions)
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'f1_macro': f1_macro
        }
    
    def train_and_evaluate(
        self,
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        experiment_name: str,
        train_source: str,
        test_source: str,
        num_epochs: int = 3,
        batch_size: int = 16,
        learning_rate: float = 2e-5,
        warmup_ratio: float = 0.1
    ) -> ValidationResult:
        """
        Train a model and evaluate on test set
        """
        print(f"\n{'='*60}")
        print(f"🔬 Experiment: {experiment_name}")
        print(f"   Train: {train_source} ({len(train_df)} samples)")
        print(f"   Test: {test_source} ({len(test_df)} samples)")
        print(f"{'='*60}")
        
        start_time = datetime.now()
        
        # Initialize tokenizer if not done
        if self.tokenizer is None:
            print("   Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        
        # Create datasets
        train_dataset = self._create_dataset(train_df)
        test_dataset = self._create_dataset(test_df)
        
        # Split training data for validation during training
        train_size = int(0.9 * len(train_dataset))
        val_size = len(train_dataset) - train_size
        train_subset, val_subset = torch.utils.data.random_split(
            train_dataset, [train_size, val_size],
            generator=torch.Generator().manual_seed(self.random_seed)
        )
        
        # Initialize fresh model
        print("   Loading model...")
        model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=len(self.label2id),
            id2label=self.id2label,
            label2id=self.label2id,
            ignore_mismatched_sizes=True
        )
        model = model.to(self.device)
        
        # Training arguments
        training_args = TrainingArguments(
            output_dir=os.path.join(self.output_dir, experiment_name),
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            warmup_ratio=warmup_ratio,
            learning_rate=learning_rate,
            weight_decay=0.01,
            logging_dir=os.path.join(self.output_dir, experiment_name, 'logs'),
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='no',  # Disable checkpoint saving to avoid I/O errors on Colab
            load_best_model_at_end=False,  # Can't load best if not saving checkpoints
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            seed=self.random_seed,
            report_to='none',
            fp16=torch.cuda.is_available(),
        )
        
        # Create Trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_subset,
            eval_dataset=val_subset,
            compute_metrics=self._compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )
        
        # Train
        print("   Training...")
        trainer.train()
        
        # Evaluate on test set
        print("   Evaluating on test set...")
        predictions = trainer.predict(test_dataset)
        preds = np.argmax(predictions.predictions, axis=1)
        labels = [self.label2id[label] for label in test_df['label']]
        
        # Calculate metrics
        accuracy = accuracy_score(labels, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average='weighted', zero_division=0
        )
        f1_macro = precision_recall_fscore_support(
            labels, preds, average='macro', zero_division=0
        )[2]
        
        # Per-class metrics
        report = classification_report(
            labels, preds,
            target_names=list(self.id2label.values()),
            output_dict=True,
            zero_division=0
        )
        
        per_class = {}
        for label_name in self.id2label.values():
            if label_name in report:
                per_class[label_name] = {
                    'precision': report[label_name]['precision'],
                    'recall': report[label_name]['recall'],
                    'f1-score': report[label_name]['f1-score'],
                    'support': report[label_name]['support']
                }
        
        # Confusion matrix
        cm = confusion_matrix(labels, preds)
        
        training_time = (datetime.now() - start_time).total_seconds()
        
        result = ValidationResult(
            experiment_name=experiment_name,
            train_source=train_source,
            test_source=test_source,
            accuracy=accuracy,
            precision_weighted=precision,
            recall_weighted=recall,
            f1_weighted=f1,
            f1_macro=f1_macro,
            per_class_metrics=per_class,
            confusion_matrix=cm,
            training_time=training_time,
            num_train_samples=len(train_df),
            num_test_samples=len(test_df)
        )
        
        self.results.append(result)
        
        print(f"\n   📈 Results:")
        print(f"      Accuracy: {accuracy:.4f}")
        print(f"      F1 (weighted): {f1:.4f}")
        print(f"      F1 (macro): {f1_macro:.4f}")
        print(f"      Training time: {training_time:.1f}s")
        
        # Cleanup
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
        return result
    
    def run_tstr(self, combined_df: pd.DataFrame, real_test_df: pd.DataFrame, **kwargs) -> ValidationResult:
        """Run TSTR: Train on Combined (Original+Augmented), Test on Real - Proves USABILITY"""
        return self.train_and_evaluate(
            train_df=combined_df,
            test_df=real_test_df,
            experiment_name='TSTR',
            train_source='Combined (Original+Augmented)',
            test_source='Real (held-out)',
            **kwargs
        )
    
    def run_trts(self, combined_df: pd.DataFrame, synth_test_df: pd.DataFrame, **kwargs) -> ValidationResult:
        """Run TRTS: Train on Combined, Test on sampled subset - Proves REALISM"""
        return self.train_and_evaluate(
            train_df=combined_df,
            test_df=synth_test_df,
            experiment_name='TRTS',
            train_source='Combined (Original+Augmented)',
            test_source='Sampled from Combined',
            **kwargs
        )


## 📈 Analysis & Reporting Functions


In [ ]:
def analyze_results(results: List[ValidationResult], label2id: Dict) -> Dict[str, Any]:
    """Analyze validation results and compute quality metrics
    
    Args:
        results: List of ValidationResult from experiments
        label2id: Label to ID mapping
    """
    if len(results) < 2:
        print("⚠️ Need at least 2 experiments (TSTR, TRTS) for analysis")
        return {}
    
    tstr = next((r for r in results if r.experiment_name == 'TSTR'), None)
    trts = next((r for r in results if r.experiment_name == 'TRTS'), None)
    
    analysis = {'summary': {}, 'gaps': {}, 'quality_assessment': {}}
    
    for r in results:
        analysis['summary'][r.experiment_name] = {
            'accuracy': r.accuracy, 'f1_weighted': r.f1_weighted,
            'f1_macro': r.f1_macro, 'train_samples': r.num_train_samples,
            'test_samples': r.num_test_samples
        }
    
    # Compute gap between TSTR and TRTS
    if tstr and trts:
        gap = abs(tstr.f1_macro - trts.f1_macro)
        analysis['gaps']['TSTR_TRTS_gap'] = gap
        analysis['gaps']['TSTR_f1'] = tstr.f1_macro
        analysis['gaps']['TRTS_f1'] = trts.f1_macro
    
    # Quality assessment for TSTR (usability on real test data) - using macro F1
    if tstr:
        tstr_f1 = tstr.f1_macro
        if tstr_f1 > 0.85:
            quality, desc = "EXCELLENT", "Combined data generalizes extremely well to real test data"
        elif tstr_f1 > 0.75:
            quality, desc = "GOOD", "Combined data is high quality with good generalization"
        elif tstr_f1 > 0.65:
            quality, desc = "MODERATE", "Noticeable gap, but combined data is usable"
        else:
            quality, desc = "POOR", "Significant gap, augmentation may need improvement"
        analysis['quality_assessment']['usability'] = {'rating': quality, 'description': desc, 'tstr_f1_macro': tstr_f1}
    
    # Quality assessment for TRTS (internal consistency) - using macro F1
    if trts:
        trts_f1 = trts.f1_macro
        if trts_f1 > 0.85:
            quality, desc = "EXCELLENT", "High internal consistency in combined data"
        elif trts_f1 > 0.75:
            quality, desc = "GOOD", "Good internal consistency with minor variations"
        elif trts_f1 > 0.65:
            quality, desc = "MODERATE", "Some inconsistency in combined data"
        else:
            quality, desc = "POOR", "Significant inconsistency in combined data"
        analysis['quality_assessment']['realism'] = {'rating': quality, 'description': desc, 'trts_f1_macro': trts_f1}
    
    return analysis


def generate_report(results: List[ValidationResult], analysis: Dict, model_name: str, output_dir: str) -> str:
    """Generate a comprehensive validation report"""
    report_lines = [
        "=" * 70, "TSTR/TRTS VALIDATION REPORT (Combined Data)",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Model: {model_name}", "=" * 70, "",
        "📊 EXPERIMENT RESULTS", "-" * 70,
        f"{'Experiment':<20} {'Accuracy':>10} {'F1-W':>10} {'F1-M':>10} {'Train':>8} {'Test':>8}",
        "-" * 70
    ]
    
    for r in results:
        report_lines.append(
            f"{r.experiment_name:<20} {r.accuracy:>10.4f} {r.f1_weighted:>10.4f} "
            f"{r.f1_macro:>10.4f} {r.num_train_samples:>8} {r.num_test_samples:>8}"
        )
    
    report_lines.extend(["", "📈 GAP ANALYSIS", "-" * 70])
    if 'gaps' in analysis:
        gaps = analysis['gaps']
        if 'TSTR_TRTS_gap' in gaps:
            report_lines.append(f"TSTR-TRTS Gap: {gaps['TSTR_TRTS_gap']:.4f}")
        if 'TSTR_f1' in gaps:
            report_lines.append(f"TSTR F1 (Combined → Real Test): {gaps['TSTR_f1']:.4f}")
        if 'TRTS_f1' in gaps:
            report_lines.append(f"TRTS F1 (Combined → Sampled Test): {gaps['TRTS_f1']:.4f}")
    
    report_lines.extend(["", "🎯 QUALITY ASSESSMENT", "-" * 70])
    if 'quality_assessment' in analysis:
        qa = analysis['quality_assessment']
        if 'usability' in qa:
            report_lines.extend([f"USABILITY (TSTR): {qa['usability']['rating']}", f"   {qa['usability']['description']}"])
        if 'realism' in qa:
            report_lines.extend([f"INTERNAL CONSISTENCY (TRTS): {qa['realism']['rating']}", f"   {qa['realism']['description']}"])
    
    report_lines.extend([
        "", "📝 INTERPRETATION GUIDE", "-" * 70,
        "• TSTR F1 > 85%: Excellent - combined data generalizes very well",
        "• TSTR F1 75-85%: Good - solid generalization to real test data",
        "• TSTR F1 65-75%: Moderate - usable but room for improvement",
        "• TSTR F1 < 65%: Poor - augmentation may need refinement",
        "", "• TRTS F1 > 85%: Excellent internal consistency",
        "• TRTS F1 75-85%: Good consistency with minor variations",
        "• TRTS F1 65-75%: Moderate - some inconsistency",
        "• TRTS F1 < 65%: Poor - significant inconsistency in data",
        "", "• Small TSTR-TRTS gap: Good balance between generalization and consistency",
        "• Large TSTR-TRTS gap: May indicate distribution issues",
        "", "=" * 70
    ])
    
    report = "\n".join(report_lines)
    print(report)
    
    # Save report and JSON
    with open(os.path.join(output_dir, 'validation_report.txt'), 'w') as f:
        f.write(report)
    
    json_results = {
        'experiments': [{'name': r.experiment_name, 'accuracy': r.accuracy, 'f1_weighted': r.f1_weighted,
                        'f1_macro': r.f1_macro, 'train_samples': r.num_train_samples, 
                        'test_samples': r.num_test_samples} for r in results],
        'analysis': analysis
    }
    with open(os.path.join(output_dir, 'validation_results.json'), 'w') as f:
        json.dump(json_results, f, indent=2)
    
    print(f"\n💾 Report saved to: {output_dir}")
    return report


In [ ]:
def plot_results(results: List[ValidationResult], id2label: Dict, output_dir: str):
    """Generate visualization plots for validation results"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Color mapping for experiments
    color_map = {
        'TSTR': '#e74c3c',           # Red - usability (real test)
        'TRTS': '#3498db',           # Blue - internal consistency
    }
    
    # 1. Comparison bar chart (Macro F1)
    ax1 = axes[0, 0]
    exp_names = [r.experiment_name for r in results]
    f1_scores = [r.f1_macro for r in results]
    colors = [color_map.get(name, '#95a5a6') for name in exp_names]
    
    bars = ax1.bar(exp_names, f1_scores, color=colors)
    ax1.set_ylabel('F1 Score (Macro)')
    ax1.set_title('TSTR/TRTS Validation Results (Macro F1)')
    ax1.set_ylim(0, 1)
    ax1.tick_params(axis='x', rotation=15)
    for bar, score in zip(bars, f1_scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{score:.3f}', ha='center', va='bottom', fontsize=10)
    
    # 2. Multi-metric comparison
    ax2 = axes[0, 1]
    x = np.arange(len(results))
    width = 0.25
    ax2.bar(x - width, [r.accuracy for r in results], width, label='Accuracy', color='#3498db')
    ax2.bar(x, [r.f1_weighted for r in results], width, label='F1 (Weighted)', color='#2ecc71')
    ax2.bar(x + width, [r.f1_macro for r in results], width, label='F1 (Macro)', color='#e74c3c')
    ax2.set_ylabel('Score')
    ax2.set_title('Multi-Metric Comparison')
    ax2.set_xticks(x)
    ax2.set_xticklabels(exp_names, rotation=15)
    ax2.legend()
    ax2.set_ylim(0, 1)
    
    # 3. Confusion matrix for TSTR
    ax3 = axes[1, 0]
    tstr = next((r for r in results if r.experiment_name == 'TSTR'), None)
    if tstr is not None:
        sns.heatmap(tstr.confusion_matrix, annot=True, fmt='d', cmap='Blues',
                   xticklabels=list(id2label.values()),
                   yticklabels=list(id2label.values()), ax=ax3)
        ax3.set_title('TSTR Confusion Matrix\n(Train Combined → Test Real)')
        ax3.set_xlabel('Predicted')
        ax3.set_ylabel('Actual')
    
    # 4. Per-class F1 comparison: TSTR vs TRTS
    ax4 = axes[1, 1]
    trts = next((r for r in results if r.experiment_name == 'TRTS'), None)
    
    if tstr and trts:
        classes = list(tstr.per_class_metrics.keys())
        tstr_f1 = [tstr.per_class_metrics.get(c, {}).get('f1-score', 0) for c in classes]
        trts_f1 = [trts.per_class_metrics.get(c, {}).get('f1-score', 0) for c in classes]
        x = np.arange(len(classes))
        width = 0.35
        ax4.bar(x - width/2, tstr_f1, width, label='TSTR (→ Real Test)', color='#e74c3c')
        ax4.bar(x + width/2, trts_f1, width, label='TRTS (→ Sampled)', color='#3498db')
        ax4.set_ylabel('F1 Score')
        ax4.set_title('Per-Class F1: TSTR vs TRTS')
        ax4.set_xticks(x)
        ax4.set_xticklabels(classes, rotation=45, ha='right')
        ax4.legend()
        ax4.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'validation_plots.png'), dpi=150, bbox_inches='tight')
    print(f"📊 Plot saved to: {output_dir}/validation_plots.png")
    plt.show()


## 🚀 Run Validation Suite

Since the data is already combined (original + augmented), this section runs 2 experiments:
1. **TSTR**: Train on Combined → Test on Real (held-out) - measures **usability/generalization**
2. **TRTS**: Train on Combined → Test on Sampled subset - measures **internal consistency**


In [ ]:
# Initialize validator
validator = TSTRTRTSValidator(
    model_name=MODEL_NAME,
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    random_seed=RANDOM_SEED
)


In [ ]:
# Load combined data (original + augmented already merged)
combined_df, test_df = validator.load_combined_data(
    combined_data_path=COMBINED_TRAIN_PATH,
    test_data_path=TEST_PATH
)

# Display data distributions
print("\n📊 Data Distributions:")
print(f"\nCombined Training Data (Original + Augmented):")
print(combined_df['label'].value_counts())
print(f"\nTest Data:")
print(test_df['label'].value_counts())


In [ ]:
# Create a sampled test set from combined data for TRTS experiment
synth_test = combined_df.sample(n=min(len(test_df), len(combined_df)), random_state=RANDOM_SEED)
print(f"Sampled test set for TRTS: {len(synth_test)} samples")


### Experiment 1: TSTR (Combined → Real Test)

Since the data is already combined (original + augmented), we train on the full combined dataset and test on held-out real test data.


In [ ]:
tstr_result = validator.run_tstr(
    combined_df=combined_df,
    real_test_df=test_df,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO
)


### Experiment 2: TRTS (Combined → Sampled Test)

Train on combined data and test on a sampled subset to check internal consistency/realism.
`

In [ ]:
trts_result = validator.run_trts(
    combined_df=combined_df,
    synth_test_df=synth_test,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO
)


## 📊 Results Analysis & Report


In [ ]:
# Analyze results (no baseline needed - using combined data)
analysis = analyze_results(validator.results, validator.label2id)

# Generate report
report = generate_report(
    results=validator.results,
    analysis=analysis,
    model_name=MODEL_NAME,
    output_dir=OUTPUT_DIR
)


## 📊 Visualization


In [ ]:
# Generate visualization plots
plot_results(
    results=validator.results,
    id2label=validator.id2label,
    output_dir=OUTPUT_DIR
)


In [ ]:
# Create summary DataFrame
summary_data = []
for r in validator.results:
    summary_data.append({
        'Experiment': r.experiment_name,
        'Train Source': r.train_source,
        'Test Source': r.test_source,
        'Accuracy': f"{r.accuracy:.4f}",
        'F1 (Weighted)': f"{r.f1_weighted:.4f}",
        'F1 (Macro)': f"{r.f1_macro:.4f}",
        'Train Samples': r.num_train_samples,
        'Test Samples': r.num_test_samples,
        'Time (s)': f"{r.training_time:.1f}"
    })

summary_df = pd.DataFrame(summary_data)
summary_df


## 🎯 Quality Assessment Summary


In [ ]:
print("\n" + "="*70)
print("🎯 COMBINED DATA QUALITY ASSESSMENT")
print("="*70)

if 'quality_assessment' in analysis:
    qa = analysis['quality_assessment']
    
    if 'usability' in qa:
        u = qa['usability']
        print(f"\n📌 USABILITY (TSTR - Combined → Real Test): {u['rating']}")
        print(f"   TSTR F1 (Macro): {u['tstr_f1_macro']:.4f}")
        print(f"   {u['description']}")
    
    if 'realism' in qa:
        r = qa['realism']
        print(f"\n📌 INTERNAL CONSISTENCY (TRTS - Combined → Sampled): {r['rating']}")
        print(f"   TRTS F1 (Macro): {r['trts_f1_macro']:.4f}")
        print(f"   {r['description']}")

if 'gaps' in analysis:
    gaps = analysis['gaps']
    if 'TSTR_TRTS_gap' in gaps:
        gap = gaps['TSTR_TRTS_gap']
        print(f"\n📌 TSTR-TRTS GAP: {gap:.4f}")
        if gap < 0.05:
            print("   ✅ Small gap - good balance between generalization and consistency")
        elif gap < 0.10:
            print("   ⚠️ Moderate gap - some distribution differences")
        else:
            print("   ❌ Large gap - may indicate distribution issues")

print("\n" + "="*70)


## 📝 Interpretation Guide

### TSTR F1 (Usability/Generalization)
- **> 85%**: EXCELLENT - Combined data generalizes very well to real test data
- **75-85%**: GOOD - Solid generalization, acceptable quality
- **65-75%**: MODERATE - Usable but room for improvement
- **< 65%**: POOR - Augmentation may need refinement

### TRTS F1 (Internal Consistency)
- **> 85%**: EXCELLENT - High internal consistency in combined data
- **75-85%**: GOOD - Good consistency with minor variations
- **65-75%**: MODERATE - Some inconsistency in data
- **< 65%**: POOR - Significant inconsistency

### TSTR-TRTS Gap
- **< 5%**: Good balance between generalization and consistency
- **5-10%**: Moderate gap, some distribution differences
- **> 10%**: Large gap, may indicate distribution issues


In [ ]:
# End of validation notebook
print("✅ TSTR/TRTS Validation Complete!")


# Keep Google Colab session alive
import time
from google.colab import runtime

def auto_disconnect():
    """Automatically disconnect Google Colab runtime after a delay"""
    time.sleep(10)  # Wait 5 minutes after completion
    print("🔌 Auto-disconnecting Colab runtime...")
    runtime.unassign()

# Start the auto-disconnect thread
if 'google.colab' in str(get_ipython()):
    import threading
    disconnect_thread = threading.Thread(target=auto_disconnect, daemon=True)
    disconnect_thread.start()
    print("⏰ Auto-disconnect scheduled (runtime will disconnect in 5 minutes)")

